In [4]:
import pandas as pd

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertModel
from transformers import AutoTokenizer

from datasets import load_dataset

from tqdm import tqdm

from sklearn.metrics import accuracy_score, classification_report, f1_score

rubert = 'DeepPavlov/rubert-base-cased'
rubert_conv = 'DeepPavlov/bert-base-cased-conversational'
bert_multilangual = 'google-bert/bert-base-multilingual-cased'
EPOCHS = 6

In [5]:
dataset_train = load_dataset('csv', data_files='qwen3_14b_labeling_clean_3211q.csv');
dataset_val = load_dataset('csv', data_files='gold_manual_anno.csv');
print(f'{dataset_train}\n\n{dataset_val}')

DatasetDict({
    train: Dataset({
        features: ['question', 'class', 'label'],
        num_rows: 3211
    })
})

DatasetDict({
    train: Dataset({
        features: ['question', 'final'],
        num_rows: 329
    })
})


In [6]:
tokenizers = {'tokenizer_rubert' : AutoTokenizer.from_pretrained(rubert),
              'tokenizer_rubert_conv' : AutoTokenizer.from_pretrained(rubert_conv),
              'tokenizer_bert_multilangual' : AutoTokenizer.from_pretrained(bert_multilangual)};

models = {'rubert' : BertForSequenceClassification.from_pretrained(rubert, num_labels=2),
          'rubert_conv' : BertForSequenceClassification.from_pretrained(rubert_conv, num_labels=2),
          'bert_multilangual' : BertForSequenceClassification.from_pretrained(bert_multilangual, num_labels=2)};

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: DeepPavlov/bert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; 

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def build_loaders(tokenizer, dataset_train, dataset_val, device, batch_size=16):

    def preprocess_function(example):
        return tokenizer(example['question'], padding='max_length', truncation=True, max_length=128)

    train_tok = dataset_train.map(preprocess_function, batched=True)
    val_tok = dataset_val.map(preprocess_function, batched=True)

    train_tok = train_tok.rename_column('label', 'labels')
    train_tok = train_tok.remove_columns(['question'])

    val_tok = val_tok.rename_column('final', 'labels')
    val_tok = val_tok.remove_columns(['question'])

    train_tok.set_format('torch');
    val_tok.set_format('torch');

    def collate_to_device(batch):
        keys = batch[0].keys()
        out = {}

        for k in keys: out[k] = torch.stack([item[k] for item in batch]).to(device)
        return out

    train_loader = DataLoader(train_tok['train'], batch_size=16, collate_fn=collate_to_device)
    val_loader = DataLoader(val_tok['train'], batch_size=16, collate_fn=collate_to_device)

    return train_loader, val_loader

In [8]:
def train_model(model, train_loader, val_loader, device, epochs=6):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    for epoch in range(EPOCHS):

        ####### TRAIN #######
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f'train epoch: {epoch+1} : ', ncols=100)

        for i, batch in enumerate(pbar):

            outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids'], labels=batch['labels'])

            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # update tqdm with %
            percent = (i + 1) / len(train_loader) * 100
            pbar.set_postfix({'progress': f'{percent:.1f}%'})

        avg_train_loss = total_loss / len(train_loader)
        print(f'train epoch: {epoch+1} : {avg_train_loss:.4f}')

        ####### VALIDATION #######
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in val_loader:
                outputs = model(input_ids=batch['input_ids'],
                            attention_mask=batch['attention_mask'],
                            token_type_ids=batch['token_type_ids'])

                preds = outputs.logits.argmax(dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(batch['labels'].cpu().tolist())

        f1 = f1_score(all_labels, all_preds, average='weighted')
        print(f'epoch: {epoch+1} validation F1: {f1:.4f}')

        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds, digits=4))


In [9]:
for model_name, model in models.items():
    print(f'\n train {model_name} \n')
    tokenizer = tokenizers[f'tokenizer_{model_name}']
    train_loader, val_loader = build_loaders(tokenizer, dataset_train, dataset_val, device)

    train_model(model, train_loader, val_loader, device, epochs=EPOCHS)


 train rubert 



Map:   0%|          | 0/3211 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

train epoch: 1 : 100%|███████████████████████████| 201/201 [01:08<00:00,  2.95it/s, progress=100.0%]


train epoch: 1 : 0.4656
epoch: 1 validation F1: 0.8319

Classification Report:
              precision    recall  f1-score   support

           0     0.7778    0.9187    0.8424       160
           1     0.9071    0.7515    0.8220       169

    accuracy                         0.8328       329
   macro avg     0.8425    0.8351    0.8322       329
weighted avg     0.8442    0.8328    0.8319       329



...

train epoch: 4 : 0.1803
epoch: 4 validation F1: 0.8293

Classification Report:
              precision    recall  f1-score   support

           0     0.7857    0.8938    0.8363       160
           1     0.8844    0.7692    0.8228       169

    accuracy                         0.8298       329
   macro avg     0.8350    0.8315    0.8295       329
weighted avg     0.8364    0.8298    0.8293       329



train epoch: 5 : 100%|███████████████████████████| 201/201 [01:13<00:00,  2.74it/s, progress=100.0%]


train epoch: 5 : 0.1259
epoch: 5 validation F1: 0.8176

Classification Report:
              precision    recall  f1-score   support

           0     0.7907    0.8500    0.8193       160
           1     0.8471    0.7870    0.8160       169

    accuracy                         0.8176       329
   macro avg     0.8189    0.8185    0.8176       329
weighted avg     0.8197    0.8176    0.8176       329



train epoch: 6 : 100%|███████████████████████████| 201/201 [01:13<00:00,  2.75it/s, progress=100.0%]


train epoch: 6 : 0.0999
epoch: 6 validation F1: 0.8228

Classification Report:
              precision    recall  f1-score   support

           0     0.8643    0.7562    0.8067       160
           1     0.7937    0.8876    0.8380       169

    accuracy                         0.8237       329
   macro avg     0.8290    0.8219    0.8223       329
weighted avg     0.8280    0.8237    0.8228       329

